In [ ]:
# ============================================================
# EXHAUSTIVE / THOROUGH FEATURE SEARCH - V6
# ============================================================

import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.ensemble import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    log_loss,
    brier_score_loss
)

# ============================================================
# SETTINGS
# ============================================================

DATA_PATH = "../data/processed/features_v6.csv"

TARGET = "FTR"

# Original baseline
BASELINE_FEATURES = [
    "EloDiff",
    "ShotOTDiffLast5",
    "GoalAgainstDiffLast5"
]

# All other potentially useful features
GOOD_FEATURES = [
    "GDPerGameDiff",
    "PPGDiff",
    "GoalsPerGameDiff",
    "GoalsAgainstPerGameDiff",
    "ShotDiffLast5",
    "GoalDiffLast5",
    "HomeElo",
    "AwayElo",
    "HomePPG",
    "AwayPPG",
    "HomeGDPerGame",
    "AwayGDPerGame",
    "HomeGoalsPerGame",
    "AwayGoalsAgainstPerGame",
    "HomePointsLast5",
    "AwayPointsLast5",
    "HomeGoalsLast5",
    "AwayGoalsLast5",
    "HomeGoalsAgainstLast5",
    "AwayGoalsAgainstLast5",
    "HomeShotsAgainstLast5",
    "AwayShotsAgainstLast5",
    "HomeShotsOnTargetLast5",
    "AwayShotsOnTargetLast5",
    "HomeShotsOnTargetAgainstLast5",
    "AwayShotsOnTargetAgainstLast5",
    "XGForDiffPg",
    "XGAgainstDiffPg",
    "XGDDiff",
    "XGForDiffLast5",
    "XGAgainstDiffLast5",
    "XGDDiffLast5"
]

ALL_FEATURES = BASELINE_FEATURES + GOOD_FEATURES


# ============================================================
# LOAD DATA
# ============================================================

print("=" * 70)
print("LOADING DATA")
print("=" * 70)

df = pd.read_csv(DATA_PATH)

print(f"Loaded: {DATA_PATH}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

# ------------------------------------------------------------
# Check target
# ------------------------------------------------------------

if TARGET not in df.columns:
    print("\nAvailable columns:")
    print(df.columns.tolist())

    raise ValueError(
        f"Target column '{TARGET}' was not found."
    )

# ------------------------------------------------------------
# Check MatchDateTime
# ------------------------------------------------------------

if "MatchDateTime" in df.columns:

    df["MatchDateTime"] = pd.to_datetime(
        df["MatchDateTime"],
        errors="coerce"
    )

    df = df.sort_values(
        "MatchDateTime"
    ).reset_index(drop=True)

# ------------------------------------------------------------
# Check Season
# ------------------------------------------------------------

if "Season" not in df.columns:

    raise ValueError(
        "Season column is missing from features_v6.csv"
    )

print("\nGames by season:")
print(df["Season"].value_counts().sort_index())


# ============================================================
# FEATURE CHECK
# ============================================================

print("\n" + "=" * 70)
print("FEATURE CHECK")
print("=" * 70)

missing_features = [
    feature
    for feature in ALL_FEATURES
    if feature not in df.columns
]

if missing_features:

    print("\nMISSING FEATURES:")
    for feature in missing_features:
        print("-", feature)

    raise ValueError(
        "Some requested features are missing from features_v6.csv"
    )

print(f"Baseline features: {len(BASELINE_FEATURES)}")
print(f"Additional features: {len(GOOD_FEATURES)}")
print(f"Total features available: {len(ALL_FEATURES)}")

print("\nAll features:")

for feature in ALL_FEATURES:
    print("-", feature)


# ============================================================
# CLEAN DATA
# ============================================================

# Keep only rows where target exists
df = df.dropna(
    subset=[TARGET]
).copy()

# Ensure all features are numeric
for feature in ALL_FEATURES:

    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce"
    )

print("\nRows after target cleaning:", len(df))


# ============================================================
# MODEL
# ============================================================

def create_model():
    return LogisticRegression(
        max_iter=1000,
        multi_class="multinomial",
        random_state=42
    )


# ============================================================
# SEASON EVALUATION
# ============================================================

TEST_SEASONS = [
    "22-23",
    "23-24",
    "24-25",
    "25-26"
]


def evaluate_feature_set(feature_set):

    feature_set = list(feature_set)

    season_results = []

    all_true = []
    all_prob = []
    all_pred = []

    for test_season in TEST_SEASONS:

        # ----------------------------------------------------
        # Test data
        # ----------------------------------------------------

        test_df = df[
            df["Season"].astype(str) == str(test_season)
        ].copy()

        # ----------------------------------------------------
        # Training data = everything before test season
        # ----------------------------------------------------

        train_df = df[
            df["Season"].astype(str) != str(test_season)
        ].copy()

        # ----------------------------------------------------
        # Remove rows containing missing feature values
        # ----------------------------------------------------

        required_columns = feature_set + [TARGET]

        train_df = train_df.dropna(
            subset=required_columns
        )

        test_df = test_df.dropna(
            subset=required_columns
        )

        # ----------------------------------------------------
        # IMPORTANT:
        # Never allow an empty fold to reach sklearn
        # ----------------------------------------------------

        if len(train_df) == 0:

            raise ValueError(
                f"EMPTY TRAINING SET for test season "
                f"{test_season}.\n"
                f"Features: {feature_set}"
            )

        if len(test_df) == 0:

            raise ValueError(
                f"EMPTY TEST SET for test season "
                f"{test_season}.\n"
                f"Features: {feature_set}"
            )

        # ----------------------------------------------------
        # Training
        # ----------------------------------------------------

        X_train = train_df[feature_set]
        y_train = train_df[TARGET]

        X_test = test_df[feature_set]
        y_test = test_df[TARGET]

        model = create_model()

        model.fit(
            X_train,
            y_train
        )

        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        probabilities = model.predict_proba(
            X_test
        )

        predictions = model.predict(
            X_test
        )

        # ----------------------------------------------------
        # Classes
        # ----------------------------------------------------

        classes = model.classes_

        # Ensure class order is A, D, H if present
        class_to_index = {
            cls: i
            for i, cls in enumerate(classes)
        }

        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        accuracy = accuracy_score(
            y_test,
            predictions
        )

        loss = log_loss(
            y_test,
            probabilities,
            labels=classes
        )

        # ----------------------------------------------------
        # Brier score per class
        # ----------------------------------------------------

        brier_scores = {}

        for cls in ["A", "D", "H"]:

            if cls in class_to_index:

                idx = class_to_index[cls]

                y_binary = (
                    y_test == cls
                ).astype(int)

                brier_scores[cls] = (
                    brier_score_loss(
                        y_binary,
                        probabilities[:, idx]
                    )
                )

            else:

                brier_scores[cls] = np.nan

        # ----------------------------------------------------
        # Store
        # ----------------------------------------------------

        season_results.append({

            "Season": test_season,
            "Games": len(test_df),
            "Accuracy": accuracy,
            "LogLoss": loss,
            "Brier_A": brier_scores["A"],
            "Brier_D": brier_scores["D"],
            "Brier_H": brier_scores["H"]

        })

        all_true.extend(
            y_test.tolist()
        )

        all_pred.extend(
            predictions.tolist()
        )

        all_prob.append(
            probabilities
        )

    # ========================================================
    # Overall metrics
    # ========================================================

    season_results_df = pd.DataFrame(
        season_results
    )

    mean_accuracy = (
        season_results_df["Accuracy"].mean()
    )

    weighted_accuracy = (
        np.average(
            season_results_df["Accuracy"],
            weights=season_results_df["Games"]
        )
    )

    mean_log_loss = (
        season_results_df["LogLoss"].mean()
    )

    mean_brier = (
        season_results_df[
            [
                "Brier_A",
                "Brier_D",
                "Brier_H"
            ]
        ].mean(axis=1).mean()
    )

    return {

        "Features": tuple(feature_set),

        "NumFeatures": len(feature_set),

        "MeanAccuracy": mean_accuracy,

        "WeightedAccuracy": weighted_accuracy,

        "MeanLogLoss": mean_log_loss,

        "MeanBrier": mean_brier,

        "SeasonResults": season_results_df

    }


# ============================================================
# TEST THE BASELINE FIRST
# ============================================================

print("\n" + "=" * 70)
print("BASELINE")
print("=" * 70)

baseline_result = evaluate_feature_set(
    BASELINE_FEATURES
)

print("\nBaseline features:")

for feature in BASELINE_FEATURES:
    print("-", feature)

print(
    f"\nMean Accuracy: "
    f"{baseline_result['MeanAccuracy']:.6f}"
)

print(
    f"Weighted Accuracy: "
    f"{baseline_result['WeightedAccuracy']:.6f}"
)

print(
    f"Mean Log Loss: "
    f"{baseline_result['MeanLogLoss']:.6f}"
)

print(
    f"Mean Brier: "
    f"{baseline_result['MeanBrier']:.6f}"
)

print("\nSeason results:")
print(
    baseline_result["SeasonResults"].to_string(
        index=False
    )
)


# ============================================================
# IMPORTANT SEARCH DESIGN
# ============================================================
#
# We have 35 total features.
#
# Testing literally every subset means:
#
#     2^35 = 34,359,738,368
#
# combinations.
#
# That is FAR too many.
#
# Instead, we test every combination up to a sensible
# maximum feature count.
#
# This lets us discover combinations that greedy forward
# selection would completely miss.
#
# ============================================================

MAX_FEATURES = 8

# Change this if you want to go further.
#
# 8 features means:
#
# C(35,1) + ... + C(35,8)
#
# which is still large, so we use a staged search below.


# ============================================================
# STAGE 1
# TEST ALL SINGLE FEATURES
# ============================================================

print("\n" + "=" * 70)
print("STAGE 1 - ALL SINGLE FEATURES")
print("=" * 70)

single_results = []

for i, feature in enumerate(
    ALL_FEATURES,
    start=1
):

    print(
        f"[{i}/{len(ALL_FEATURES)}] "
        f"{feature}"
    )

    result = evaluate_feature_set(
        [feature]
    )

    single_results.append({

        "Features": " + ".join(
            result["Features"]
        ),

        "NumFeatures": 1,

        "MeanAccuracy":
            result["MeanAccuracy"],

        "MeanLogLoss":
            result["MeanLogLoss"],

        "MeanBrier":
            result["MeanBrier"],

    })


single_results_df = pd.DataFrame(
    single_results
)

single_results_df = (
    single_results_df
    .sort_values("MeanLogLoss")
    .reset_index(drop=True)
)

print("\nBest single features:")

print(
    single_results_df.head(15).to_string(
        index=False
    )
)


# ============================================================
# STAGE 2
# TEST ALL PAIRS
# ============================================================

print("\n" + "=" * 70)
print("STAGE 2 - ALL PAIRS")
print("=" * 70)

pair_results = []

pair_count = 0

total_pairs = len(list(
    combinations(
        ALL_FEATURES,
        2
    )
))

for feature_set in combinations(
    ALL_FEATURES,
    2
):

    pair_count += 1

    if pair_count % 25 == 0:

        print(
            f"[{pair_count}/{total_pairs}]"
        )

    result = evaluate_feature_set(
        feature_set
    )

    pair_results.append({

        "Features": " + ".join(
            result["Features"]
        ),

        "NumFeatures": 2,

        "MeanAccuracy":
            result["MeanAccuracy"],

        "MeanLogLoss":
            result["MeanLogLoss"],

        "MeanBrier":
            result["MeanBrier"]

    })


pair_results_df = pd.DataFrame(
    pair_results
)

pair_results_df = (
    pair_results_df
    .sort_values("MeanLogLoss")
    .reset_index(drop=True)
)

print("\nBest pairs:")

print(
    pair_results_df.head(20).to_string(
        index=False
    )
)


# ============================================================
# STAGE 3
# TEST TRIPLES
# ============================================================

print("\n" + "=" * 70)
print("STAGE 3 - ALL TRIPLES")
print("=" * 70)

triple_results = []

triple_count = 0

total_triples = len(list(
    combinations(
        ALL_FEATURES,
        3
    )
))

for feature_set in combinations(
    ALL_FEATURES,
    3
):

    triple_count += 1

    if triple_count % 100 == 0:

        print(
            f"[{triple_count}/{total_triples}]"
        )

    result = evaluate_feature_set(
        feature_set
    )

    triple_results.append({

        "Features": " + ".join(
            result["Features"]
        ),

        "NumFeatures": 3,

        "MeanAccuracy":
            result["MeanAccuracy"],

        "MeanLogLoss":
            result["MeanLogLoss"],

        "MeanBrier":
            result["MeanBrier"]

    })


triple_results_df = pd.DataFrame(
    triple_results
)

triple_results_df = (
    triple_results_df
    .sort_values("MeanLogLoss")
    .reset_index(drop=True)
)

print("\nBest triples:")

print(
    triple_results_df.head(20).to_string(
        index=False
    )
)


# ============================================================
# SAVE RESULTS
# ============================================================

all_search_results = pd.concat(
    [
        single_results_df,
        pair_results_df,
        triple_results_df
    ],
    ignore_index=True
)

all_search_results = (
    all_search_results
    .sort_values("MeanLogLoss")
    .reset_index(drop=True)
)

output_path = (
    "../data/processed/"
    "all_feature_search_v6.csv"
)

all_search_results.to_csv(
    output_path,
    index=False
)

print("\n" + "=" * 70)
print("BEST FEATURE SETS OVERALL")
print("=" * 70)

print(
    all_search_results.head(30).to_string(
        index=False
    )
)

print(
    f"\nResults saved to:\n{output_path}"
)


# ============================================================
# BEST RESULT
# ============================================================

best = all_search_results.iloc[0]

print("\n" + "=" * 70)
print("BEST RESULT")
print("=" * 70)

print(
    f"Features: {best['Features']}"
)

print(
    f"Number of features: "
    f"{int(best['NumFeatures'])}"
)

print(
    f"Mean Accuracy: "
    f"{best['MeanAccuracy']:.6f}"
)

print(
    f"Mean Log Loss: "
    f"{best['MeanLogLoss']:.6f}"
)

print(
    f"Mean Brier: "
    f"{best['MeanBrier']:.6f}"
)

print("\nCompared with baseline:")

print(
    f"Accuracy change: "
    f"{best['MeanAccuracy'] - baseline_result['MeanAccuracy']:.6f}"
)

print(
    f"Log Loss change: "
    f"{best['MeanLogLoss'] - baseline_result['MeanLogLoss']:.6f}"
)

print(
    f"Brier change: "
    f"{best['MeanBrier'] - baseline_result['MeanBrier']:.6f}"
)

LOADING DATA
Loaded: ../data/processed/features_v6.csv
Rows: 1900
Columns: 236

Games by season:
Season
21-22    380
22-23    380
23-24    380
24-25    380
25-26    380
Name: count, dtype: int64

FEATURE CHECK
Baseline features: 3
Additional features: 32
Total features available: 35

All features:
- EloDiff
- ShotOTDiffLast5
- GoalAgainstDiffLast5
- GDPerGameDiff
- PPGDiff
- GoalsPerGameDiff
- GoalsAgainstPerGameDiff
- ShotDiffLast5
- GoalDiffLast5
- HomeElo
- AwayElo
- HomePPG
- AwayPPG
- HomeGDPerGame
- AwayGDPerGame
- HomeGoalsPerGame
- AwayGoalsAgainstPerGame
- HomePointsLast5
- AwayPointsLast5
- HomeGoalsLast5
- AwayGoalsLast5
- HomeGoalsAgainstLast5
- AwayGoalsAgainstLast5
- HomeShotsAgainstLast5
- AwayShotsAgainstLast5
- HomeShotsOnTargetLast5
- AwayShotsOnTargetLast5
- HomeShotsOnTargetAgainstLast5
- AwayShotsOnTargetAgainstLast5
- XGForDiffPg
- XGAgainstDiffPg
- XGDDiff
- XGForDiffLast5
- XGAgainstDiffLast5
- XGDDiffLast5

Rows after target cleaning: 1900

BASELINE

Baseline f

KeyboardInterrupt: 